# 📊 NB3 — Feature Engineering
**Scotiabank Perú · Prevención de Fraude**

Variables a construir:
- Bloque A: Velocidad por cuenta origen
- Bloque B: Relación origen → destino (beneficiario nuevo)
- Bloque C: Cuenta mula (destino que recibe de múltiples orígenes)
- Bloque D: Historial del cliente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

# ── AJUSTAR ESTOS NOMBRES ──────────────────────────────────────
COL_MONTO    = 'MONTO'
COL_IND      = 'INDICADOR'
COL_SEGMENTO = 'SEGMENTO'
COL_FECHA    = 'FECHA_HORA'          # datetime combinado
COL_ORIGEN   = 'ACF_CUENTA_ORIGEN'   # <-- reemplaza con nombre real
COL_DESTINO  = 'ACF_CUENTA_DESTINO'  # <-- reemplaza con nombre real
COL_CLIENTE  = 'ACFYD_CLIENTE'       # <-- reemplaza con nombre real
COL_BENEF    = 'CF IP-NOMBRE CIUDAD' # <-- reemplaza con nombre real
# ──────────────────────────────────────────────────────────────

COLORES = {'F':'#E63946','N':'#457B9D','G':'#2A9D8F'}

SEG_NOMBRE = {
    '30':'Polo Dirección','99':'Polo Dirección',
    '31':'Premium',       '32':'Preferente',
    '33':'Personal',      '34':'Estándar',
    '5' :'Inst. Financieras','21':'Corporativo',
    '2' :'Mediano Empresas','15':'Sector Gobierno',
    '16':'Otras Instituciones',
    '3' :'Pequeñas Empresas','4':'Negocios 2',
    '7' :'Negocios 3',    '8':'Negocios 1',
    '13':'Microempresas',
}

print('✅ Config lista')

In [ ]:
# ── Carga ──────────────────────────────────────────────────────
df = pd.read_parquet('transferencias_consolidado.parquet')
df[COL_IND]      = df[COL_IND].astype(str).str.strip().str.upper()
df[COL_SEGMENTO] = df[COL_SEGMENTO].astype(str).str.strip()
df[COL_ORIGEN]   = df[COL_ORIGEN].astype(str).str.strip()
df[COL_DESTINO]  = df[COL_DESTINO].astype(str).str.strip()
df[COL_CLIENTE]  = df[COL_CLIENTE].astype(str).str.strip()

# Normalizar nombre beneficiario
df['NOMBRE_BENEFICIARIO'] = (
    df[COL_BENEF]
    .astype(str).str.strip().str.upper()
    .str.replace(r'\s+', ' ', regex=True)
    .replace('NAN', 'SIN_NOMBRE')
    .fillna('SIN_NOMBRE')
)

# Ordenar por cuenta origen y fecha — indispensable para ventanas
df = df.sort_values([COL_ORIGEN, COL_FECHA]).reset_index(drop=True)

print(f'Dataset: {len(df):,} filas')
print(f'Cuentas origen únicas  : {df[COL_ORIGEN].nunique():,}')
print(f'Cuentas destino únicas : {df[COL_DESTINO].nunique():,}')
print(f'Clientes únicos        : {df[COL_CLIENTE].nunique():,}')

## Bloque A — Velocidad por cuenta origen

In [ ]:
# Conteo de transacciones y monto acumulado por cuenta por día
# (ventana diaria es lo más estable con datos quincenales)

vel_dia = (
    df.groupby([COL_ORIGEN, 'HORA', df[COL_FECHA].dt.date])
    .agg(
        VEL_TRX_DIA   = (COL_MONTO, 'count'),
        VEL_MONTO_DIA = (COL_MONTO, 'sum')
    )
    .reset_index()
    .rename(columns={COL_FECHA: 'FECHA_DIA'})
)

# Merge de vuelta al dataframe principal
df['FECHA_DIA'] = df[COL_FECHA].dt.date
df = df.merge(
    vel_dia[[COL_ORIGEN, 'FECHA_DIA', 'VEL_TRX_DIA', 'VEL_MONTO_DIA']],
    on=[COL_ORIGEN, 'FECHA_DIA'],
    how='left'
)

# Flags de velocidad
df['FLAG_VEL_ALTA']   = (df['VEL_TRX_DIA'] >= 2).astype(int)   # 2+ txn mismo día
df['FLAG_VEL_CRITICA'] = (df['VEL_TRX_DIA'] >= 3).astype(int)  # 3+ txn mismo día

print('Velocidad construida:')
print(f"  FLAG_VEL_ALTA   (≥2 txn/día): {df['FLAG_VEL_ALTA'].sum():,} transacciones")
print(f"  FLAG_VEL_CRITICA (≥3 txn/día): {df['FLAG_VEL_CRITICA'].sum():,} transacciones")

# ¿Discrimina en fraudes?
check = df[df[COL_IND].isin(['F','N','G'])]
print('\n% FLAG_VEL_ALTA por indicador:')
print(check.groupby(COL_IND)['FLAG_VEL_ALTA'].mean().mul(100).round(2))

In [ ]:
# Velocidad acumulada por semana (VEL_TRX_7D)
df['SEMANA'] = df[COL_FECHA].dt.to_period('W')

vel_semana = (
    df.groupby([COL_ORIGEN, 'SEMANA'])
    .agg(
        VEL_TRX_7D   = (COL_MONTO, 'count'),
        VEL_MONTO_7D = (COL_MONTO, 'sum')
    )
    .reset_index()
)

df = df.merge(
    vel_semana[[COL_ORIGEN, 'SEMANA', 'VEL_TRX_7D', 'VEL_MONTO_7D']],
    on=[COL_ORIGEN, 'SEMANA'],
    how='left'
)

print('Velocidad semanal construida ✅')
print(f"  VEL_TRX_7D media : {df['VEL_TRX_7D'].mean():.2f} txn/semana por cuenta")
print(f"  VEL_TRX_7D en F  : {df.loc[df[COL_IND]=='F','VEL_TRX_7D'].mean():.2f}")
print(f"  VEL_TRX_7D en N  : {df.loc[df[COL_IND]=='N','VEL_TRX_7D'].mean():.2f}")

## Bloque B — Relación origen → destino (beneficiario nuevo)

In [ ]:
# ¿Cuántas veces este par origen→destino ha operado antes?
# Ordenado por fecha, contamos transacciones previas del mismo par

df = df.sort_values([COL_ORIGEN, COL_DESTINO, COL_FECHA]).reset_index(drop=True)

df['FREC_PAR_ACUM'] = (
    df.groupby([COL_ORIGEN, COL_DESTINO]).cumcount()  # 0 = primera vez
)

# Flag: primera vez que origen transfiere a este destino
df['FLAG_BENEF_NUEVO'] = (df['FREC_PAR_ACUM'] == 0).astype(int)

print('Beneficiario nuevo construido:')
check = df[df[COL_IND].isin(['F','N','G'])]
print('\n% FLAG_BENEF_NUEVO por indicador:')
resumen_benef = check.groupby(COL_IND)['FLAG_BENEF_NUEVO'].mean().mul(100).round(2)
print(resumen_benef)

sep = resumen_benef.get('F',0) - resumen_benef.get('N',0)
if sep > 5:
    print(f'\n→ FLAG_BENEF_NUEVO discrimina (+{sep:.1f}pp en F vs N) ✅')
else:
    print(f'\n→ FLAG_BENEF_NUEVO no discrimina ({sep:.1f}pp) ❌')

## Bloque C — Cuenta mula (destino que recibe de múltiples orígenes)

In [ ]:
# ¿Cuántas cuentas origen DISTINTAS le transfirieron a este destino
# en el mismo día? → señal de cuenta mula

mula_dia = (
    df.groupby([COL_DESTINO, 'FECHA_DIA'])[COL_ORIGEN]
    .nunique()
    .reset_index()
    .rename(columns={COL_ORIGEN: 'ORIGENES_DIA'})
)

df = df.merge(mula_dia, on=[COL_DESTINO, 'FECHA_DIA'], how='left')
df['FLAG_CUENTA_MULA'] = (df['ORIGENES_DIA'] >= 3).astype(int)  # 3+ orígenes distintos

# Mismo análisis en ventana semanal
mula_semana = (
    df.groupby([COL_DESTINO, 'SEMANA'])[COL_ORIGEN]
    .nunique()
    .reset_index()
    .rename(columns={COL_ORIGEN: 'ORIGENES_SEMANA'})
)
df = df.merge(mula_semana, on=[COL_DESTINO, 'SEMANA'], how='left')

print('Cuenta mula construida:')
print(f"  Cuentas destino con ≥3 orígenes/día: "
      f"{df.loc[df['FLAG_CUENTA_MULA']==1, COL_DESTINO].nunique():,}")

check = df[df[COL_IND].isin(['F','N','G'])]
print('\n% FLAG_CUENTA_MULA por indicador:')
resumen_mula = check.groupby(COL_IND)['FLAG_CUENTA_MULA'].mean().mul(100).round(2)
print(resumen_mula)

sep = resumen_mula.get('F',0) - resumen_mula.get('N',0)
if sep > 5:
    print(f'\n→ FLAG_CUENTA_MULA discrimina (+{sep:.1f}pp en F vs N) ✅')
else:
    print(f'\n→ FLAG_CUENTA_MULA no discrimina ({sep:.1f}pp) ❌')

In [ ]:
# Top cuentas destino sospechosas — las que más orígenes distintos reciben
top_mulas = (
    df.groupby(COL_DESTINO)
    .agg(
        ORIGENES_TOTALES    = (COL_ORIGEN,  'nunique'),
        N_TRANSACCIONES     = (COL_MONTO,   'count'),
        MONTO_RECIBIDO      = (COL_MONTO,   'sum'),
        N_FRAUDES_ASOCIADOS = (COL_IND,     lambda x: (x=='F').sum()),
        NOMBRE_BENEF        = ('NOMBRE_BENEFICIARIO', 'first')
    )
    .sort_values('ORIGENES_TOTALES', ascending=False)
    .head(20)
    .reset_index()
)
top_mulas['MONTO_RECIBIDO'] = top_mulas['MONTO_RECIBIDO'].map(lambda x: f'S/ {x:,.0f}')

print('🚨 TOP 20 CUENTAS DESTINO SOSPECHOSAS (mayor cantidad de orígenes distintos):')
display(top_mulas)

## Bloque D — Historial del cliente

In [ ]:
# ¿El cliente ya tuvo un fraude confirmado antes en el período?
df = df.sort_values([COL_CLIENTE, COL_FECHA]).reset_index(drop=True)

# Marcar fraudes acumulados por cliente (excluyendo la fila actual)
df['FRAUDES_PREVIOS_CLIENTE'] = (
    df.groupby(COL_CLIENTE)[COL_IND]
    .transform(lambda x: (x == 'F').shift(1).fillna(0).cumsum())
    .astype(int)
)
df['FLAG_CLIENTE_REINCIDENTE'] = (df['FRAUDES_PREVIOS_CLIENTE'] >= 1).astype(int)

# Días desde último fraude del cliente
fechas_fraude = df.loc[df[COL_IND]=='F', [COL_CLIENTE, COL_FECHA]].copy()
fechas_fraude = fechas_fraude.rename(columns={COL_FECHA: 'FECHA_ULTIMO_FRAUDE'})
fechas_fraude = fechas_fraude.sort_values([COL_CLIENTE,'FECHA_ULTIMO_FRAUDE'])

# Último fraude por cliente hasta cada fecha
df = df.merge(
    fechas_fraude.groupby(COL_CLIENTE)['FECHA_ULTIMO_FRAUDE'].max().reset_index(),
    on=COL_CLIENTE, how='left'
)
df['DIAS_DESDE_ULTIMO_FRAUDE'] = (
    (df[COL_FECHA] - df['FECHA_ULTIMO_FRAUDE']).dt.days
)

print('Historial cliente construido:')
reinc = df[df[COL_IND].isin(['F','N','G'])]
print(f"\n  Clientes reincidentes: {df['FLAG_CLIENTE_REINCIDENTE'].sum():,}")
print('\n% FLAG_CLIENTE_REINCIDENTE por indicador:')
print(reinc.groupby(COL_IND)['FLAG_CLIENTE_REINCIDENTE'].mean().mul(100).round(2))

## Resumen — discriminación de cada variable nueva

In [ ]:
check = df[df[COL_IND].isin(['F','N','G'])].copy()

FLAGS = [
    ('FLAG_VEL_ALTA',          '≥2 txn mismo día (cuenta origen)'),
    ('FLAG_VEL_CRITICA',       '≥3 txn mismo día (cuenta origen)'),
    ('FLAG_BENEF_NUEVO',       'Primera vez origen → ese destino'),
    ('FLAG_CUENTA_MULA',       '≥3 orígenes distintos → mismo destino/día'),
    ('FLAG_CLIENTE_REINCIDENTE','Cliente con fraude previo confirmado'),
]

filas = []
for col, desc in FLAGS:
    if col not in check.columns:
        continue
    pct_F = check.loc[check[COL_IND]=='F', col].mean() * 100
    pct_N = check.loc[check[COL_IND]=='N', col].mean() * 100
    sep   = pct_F - pct_N
    filas.append({
        'Variable'   : col,
        'Descripción': desc,
        '% en F'     : f'{pct_F:.1f}%',
        '% en N'     : f'{pct_N:.1f}%',
        'Separación' : f'{sep:+.1f}pp',
        'Útil'       : '✅' if sep > 5 else ('⚠️' if sep > 2 else '❌')
    })

resumen_flags = pd.DataFrame(filas)
print('\n📋 RESUMEN DE DISCRIMINACIÓN — VARIABLES DE COMPORTAMIENTO')
print('='*75)
display(resumen_flags)
print('\n→ Variables ✅ son candidatas para las reglas del NB4')

## Guardar dataset enriquecido

In [ ]:
# Guardar con todas las variables nuevas para usar en NB4
df.to_parquet('transferencias_features.parquet', index=False)
print('✅ Dataset enriquecido guardado: transferencias_features.parquet')
print(f'   Columnas totales: {df.shape[1]}')
print('\nVariables nuevas agregadas:')
vars_nuevas = [
    'VEL_TRX_DIA','VEL_MONTO_DIA','FLAG_VEL_ALTA','FLAG_VEL_CRITICA',
    'VEL_TRX_7D','VEL_MONTO_7D',
    'FREC_PAR_ACUM','FLAG_BENEF_NUEVO',
    'ORIGENES_DIA','ORIGENES_SEMANA','FLAG_CUENTA_MULA',
    'FRAUDES_PREVIOS_CLIENTE','FLAG_CLIENTE_REINCIDENTE',
    'DIAS_DESDE_ULTIMO_FRAUDE'
]
for v in vars_nuevas:
    existe = '✅' if v in df.columns else '❌'
    print(f'  {existe} {v}')